# 02 — Evidence & Decisions

Run this **after** every training run, before touching `logs/EXPERIMENTS.md`.

Its job is to answer one question honestly: *did that change actually help, or did
a number move by chance?* The rules it enforces are in [AGENTS.md](../AGENTS.md).

No CPU/GPU needed — this reads saved out-of-fold predictions, so it runs fine on a
CPU runtime while a training job holds the GPU elsewhere.

## Setup

In [ ]:
import os, sys
from pathlib import Path

REPO = '/content/OctWave3'
if os.path.exists(REPO):                       # Colab
    !cd $REPO && git pull -q
    sys.path.insert(0, REPO)
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    OUT = Path('/content/drive/MyDrive/octwave3/outputs')
else:                                          # local
    sys.path.insert(0, str(Path.cwd().parent))
    OUT = Path('../outputs')

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.config import cfg
from src.analysis import *
from src.utils import load_oof

cfg.out_dir = OUT
use_house_style()
print('OOF files:', sorted(p.name for p in (OUT / 'oof').glob('*.npz')))

## 1. Single experiment — the four-panel report

Learning curves (overfitting), fold spread, confusion matrix (where errors are),
and reliability (are the probabilities honest). Title carries the 95% CI, because
a point estimate on its own invites over-reading.

In [ ]:
EXP    = 'exp01_baseline'
FOLDS  = [0]                                   # extend as you train more folds
classes = [str(i) for i in range(cfg.num_classes)]   # replace with real class names

run_log = pd.read_json(OUT / 'run_log.jsonl', lines=True)
fig = evidence_report(cfg, EXP, FOLDS, classes, run_log,
                      out_png=OUT / f'figures/{EXP}.png')

## 2. How uncertain is that score?

If a rival model's score falls inside this interval, you have no evidence they differ.

In [ ]:
probs, targets = load_oof(OUT, EXP, FOLDS[0])
for m in ['accuracy', 'macro_f1']:
    ci = bootstrap_ci(targets, probs, metric=m)
    print(f"{m:>9}: {ci['point']:.4f}  95% CI [{ci['lo']:.4f}, {ci['hi']:.4f}]  (width {ci['width']:.4f})")

## 3. Is the candidate actually better?

**The gate.** Paired bootstrap + McNemar on the same validation samples.
Only an `ADOPT` verdict makes the candidate the new baseline.

In [ ]:
BASE = 'exp01_baseline'
CAND = 'exp02_b3_300px'
FOLD = 0

probs_a, targets = load_oof(OUT, BASE, FOLD)
probs_b, _       = load_oof(OUT, CAND, FOLD)

result = decide(targets, probs_a, probs_b, BASE, CAND, metric='accuracy')

## 4. Fold spread — the noise band made visible

Heavily overlapping boxes mean the difference in means is not real.

In [ ]:
summaries = [cv_summary(OUT, e, FOLDS) for e in [BASE, CAND]]
plot_fold_box(summaries)
plt.show()
for s in summaries:
    print(f"{s['exp']:>22}: {s['mean']:.4f} ± {s['std']:.4f}   folds={np.round(s['scores'], 4)}")

## 5. Where are the errors?

Drives the *next* experiment: a hot off-diagonal cell is a targeted fix, not 'train longer'.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
plot_confusion(targets, probs, classes, ax=ax[0])
plot_per_class_f1(targets, probs, classes, ax=ax[1])
plt.tight_layout(); plt.show()

## 6. Can I trust my CV?

**The most important chart in the competition.** Fill in `logs/EXPERIMENTS.md` as
you submit, then plot CV against public LB.

- `r > 0.7` → CV is trustworthy; iterate offline and stop burning submissions.
- low / negative `r` → the validation split is wrong. Fix that before tuning anything.

In [ ]:
exps = pd.DataFrame([
    # {'exp': 'exp01', 'cv': 0.0000, 'lb': 0.0000},
])
if len(exps):
    plot_cv_vs_lb(exps); plt.show()
else:
    print('no submissions logged yet - fill this in from logs/EXPERIMENTS.md')

## 7. Record it

Paste the `decide()` output into `logs/DECISIONS.md` using the template there, add
a row to `logs/EXPERIMENTS.md`, and commit. An unlogged experiment is a wasted one —
you will otherwise re-run it in three days having forgotten the result.